In [ ]:
import math
from IPython.display import display, clear_output
import ipywidgets as widgets

# -------------------------------------------------------------- Funciones para Interés Simple --------------------------------------------------------------
def interes_simple(principal, tasa, tiempo):
    "Calcula el interés simple: I=P*r*t"
    return principal * tasa * tiempo / 100

def valor_futuro_simple(principal, tasa, tiempo):
    "Calcula el valor futuro: VF = P + I"
    return principal + interes_simple(principal, tasa, tiempo)

def valor_presente_simple(valor_futuro, tasa, tiempo):
    "Calcula el valor presente: VP = VF / (1 + r * t)"
    denominador = 1 + tasa * tiempo / 100
    if denominador == 0:
        raise ValueError("El denominador no puede ser cero")
    return valor_futuro / denominador

def tasa_interes_simple(valor_futuro, principal, tiempo):
    "Calcula la tasa de interés: r = ((VF / P) - 1) / t * 100"
    if tiempo == 0:
        raise ValueError("El tiempo no puede ser cero")
    if principal == 0:
        raise ValueError("El capital no puede ser cero")
    return ((valor_futuro / principal) - 1) / tiempo * 100

def tiempo_interes_simple(valor_futuro, principal, tasa):
    "Calcula el tiempo: t = ((VF / P) - 1) / r"
    if tasa == 0:
        raise ValueError("La tasa no puede ser cero")
    if principal == 0:
        raise ValueError("El capital no puede ser cero")
    return ((valor_futuro / principal) - 1) / (tasa / 100)

# -------------------------------------------------------------- Funciones para Interés Compuesto --------------------------------------------------------------
def valor_futuro_compuesto(principal, tasa, tiempo, n=1):
    "Calcula el valor futuro: VF = P * (1 + r/n)^(n*t)"
    if n == 0:
        raise ValueError("El número de periodos por año no puede ser cero")
    return principal * (1 + tasa / (100 * n)) ** (n * tiempo)

def valor_presente_compuesto(valor_futuro, tasa, tiempo, n=1):
    "Calcula el valor presente: VP = VF / (1 + r/n)^(n*t)"
    if n == 0:
        raise ValueError("El número de periodos por año no puede ser cero")
    denominador = (1 + tasa / (100 * n)) ** (n * tiempo)
    if denominador == 0:
        raise ValueError("El denominador no puede ser cero")
    return valor_futuro / denominador

def tiempo_interes_compuesto(valor_futuro, principal, tasa, n=1):
    "Calcula el tiempo: t = ln(VF / P) / (n * ln(1 + r/n))"
    if n == 0:
        raise ValueError("El número de periodos por año no puede ser cero")
    if principal == 0:
        raise ValueError("El capital no puede ser cero")
    if tasa == 0:
        raise ValueError("La tasa no puede ser cero")
    if valor_futuro / principal <= 0:
        raise ValueError("El cociente VF/P debe ser positivo")
    denominador = n * math.log(1 + tasa / (100 * n))
    if denominador == 0:
        raise ValueError("El denominador no puede ser cero")
    return math.log(valor_futuro / principal) / denominador

# -------------------------------------------------------------- Función principal para la calculadora --------------------------------------------------------------
def calculadora_financiera():
    # Widgets para selección de tema y cálculo
    tema = widgets.Dropdown(
        options=['Interés Simple', 'Interés Compuesto'],
        description='Tema:',
        style={'description_width': 'initial'}
    )
    
    calculo = widgets.Dropdown(
        options=[],
        description='Cálculo:',
        style={'description_width': 'initial'}
    )
    
    # Widgets para entradas numéricas
    principal = widgets.FloatText(description='Capital ($):', value=0, style={'description_width': 'initial'})
    valor_futuro = widgets.FloatText(description='Valor Futuro ($):', value=0, style={'description_width': 'initial'})
    tasa = widgets.FloatText(description='Tasa (%):', value=0, style={'description_width': 'initial'})
    tiempo = widgets.FloatText(description='Tiempo (años):', value=0, style={'description_width': 'initial'})
    periodos = widgets.IntText(description='Periodos por año:', value=1, style={'description_width': 'initial'})
    
    # Widget para mostrar el resultado
    resultado = widgets.Output()
    
    # Función para actualizar las opciones de cálculo según el tema
    def actualizar_opciones_calculo(cambio):
        if cambio['new'] == 'Interés Simple':
            calculo.options = [
                'Interés', 'Valor Futuro', 'Valor Presente', 'Tasa de Interés', 'Tiempo'
            ]
        else:
            calculo.options = [
                'Valor Futuro', 'Valor Presente', 'Tiempo'
            ]
        calculo.value = calculo.options[0] if calculo.options else None
        # Limpiar entradas
        principal.value = 0
        valor_futuro.value = 0
        tasa.value = 0
        tiempo.value = 0
        periodos.value = 1
        with resultado:
            clear_output()
    
    # Asignar la función de actualización al cambio de tema
    tema.observe(actualizar_opciones_calculo, names='value')
    
    # Función para realizar el cálculo
    def calcular(_):
        with resultado:
            clear_output()
            try:
                # Validar entradas no negativas
                if (principal.value < 0 or valor_futuro.value < 0 or 
                    tasa.value < 0 or tiempo.value < 0 or periodos.value < 0):
                    print("Error: Los valores no pueden ser negativos")
                    return
                
                if tema.value == 'Interés Simple':
                    if calculo.value == 'Interés':
                        if principal.value == 0:
                            print("Error: El capital debe ser mayor que cero")
                            return
                        res = interes_simple(principal.value, tasa.value, tiempo.value)
                        print(f"Interés calculado: ${res:,.2f}")
                    elif calculo.value == 'Valor Futuro':
                        if principal.value == 0:
                            print("Error: El capital debe ser mayor que cero")
                            return
                        res = valor_futuro_simple(principal.value, tasa.value, tiempo.value)
                        print(f"Valor Futuro calculado: ${res:,.2f}")
                    elif calculo.value == 'Valor Presente':
                        if valor_futuro.value == 0:
                            print("Error: El valor futuro debe ser mayor que cero")
                            return
                        res = valor_presente_simple(valor_futuro.value, tasa.value, tiempo.value)
                        print(f"Valor Presente calculado: ${res:,.2f}")
                    elif calculo.value == 'Tasa de Interés':
                        res = tasa_interes_simple(valor_futuro.value, principal.value, tiempo.value)
                        print(f"Tasa de interés calculada: {res:.2f}%")
                    elif calculo.value == 'Tiempo':
                        res = tiempo_interes_simple(valor_futuro.value, principal.value, tasa.value)
                        print(f"Tiempo calculado: {res:.2f} años")
                else:  # Interés Compuesto
                    if calculo.value == 'Valor Futuro':
                        if principal.value == 0:
                            print("Error: El capital debe ser mayor que cero")
                            return
                        res = valor_futuro_compuesto(principal.value, tasa.value, tiempo.value, periodos.value)
                        print(f"Valor Futuro compuesto: ${res:,.2f}")
                    elif calculo.value == 'Valor Presente':
                        if valor_futuro.value == 0:
                            print("Error: El valor futuro debe ser mayor que cero")
                            return
                        res = valor_presente_compuesto(valor_futuro.value, tasa.value, tiempo.value, periodos.value)
                        print(f"Valor Presente compuesto: ${res:,.2f}")
                    elif calculo.value == 'Tiempo':
                        if valor_futuro.value == 0:
                            print("Error: El valor futuro debe ser mayor que cero")
                            return
                        res = tiempo_interes_compuesto(valor_futuro.value, principal.value, tasa.value, periodos.value)
                        print(f"⏳ Tiempo calculado: {res:.2f} años")
            except ValueError as e:
                print(f"Error: {str(e)}")
            except Exception as e:
                print(f"Error inesperado: {str(e)}")
    
    # Botón para calcular
    boton = widgets.Button(description='Calcular', button_style='success', tooltip='Realizar cálculo')
    boton.on_click(calcular)
    
    # Mostrar widgets según el cálculo seleccionado
    def actualizar_entradas(cambio):
        entradas = [tema, calculo]
        if tema.value == 'Interés Simple':
            if calculo.value in ['Interés', 'Valor Futuro']:
                entradas += [principal, tasa, tiempo]
            elif calculo.value == 'Valor Presente':
                entradas += [valor_futuro, tasa, tiempo]
            elif calculo.value == 'Tasa de Interés':
                entradas += [valor_futuro, principal, tiempo]
            elif calculo.value == 'Tiempo':
                entradas += [valor_futuro, principal, tasa]
        else:  # Interés Compuesto
            if calculo.value == 'Valor Futuro':
                entradas += [principal, tasa, tiempo, periodos]
            elif calculo.value == 'Valor Presente':
                entradas += [valor_futuro, tasa, tiempo, periodos]
            elif calculo.value == 'Tiempo':
                entradas += [valor_futuro, principal, tasa, periodos]
        entradas += [boton, resultado]
        clear_output()
        display(*entradas)
    
    # Asignar la función de actualización al cambio de tema y cálculo
    tema.observe(actualizar_entradas, names='value')
    calculo.observe(actualizar_entradas, names='value')
    
    # Inicializar la interfaz
    actualizar_opciones_calculo({'new': tema.value})
    display(tema, calculo)

# Ejecutar la calculadora
calculadora_financiera()

Dropdown(description='Tema:', index=1, options=('Interés Simple', 'Interés Compuesto'), style=DescriptionStyle…

Dropdown(description='Cálculo:', index=2, options=('Valor Futuro', 'Valor Presente', 'Tiempo'), style=Descript…

FloatText(value=1000.0, description='Valor Futuro ($):', style=DescriptionStyle(description_width='initial'))

FloatText(value=500.0, description='Capital ($):', style=DescriptionStyle(description_width='initial'))

FloatText(value=6.0, description='Tasa (%):', style=DescriptionStyle(description_width='initial'))

IntText(value=2, description='Periodos por año:', style=DescriptionStyle(description_width='initial'))

Button(button_style='success', description='Calcular', style=ButtonStyle(), tooltip='Realizar cálculo')

Output(outputs=({'name': 'stdout', 'text': '📉 Valor Presente compuesto: $701.38\n', 'output_type': 'stream'},)…